In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# -------------------------------
# CSS for card-style panels & responsive vertical layout
# -------------------------------
display(HTML("""
<style>
.panel {
    border: 1px solid #ccc;
    border-radius: 8px;
    padding: 8px;
    margin-bottom: 12px;
    background-color: #fafafa;
    box-shadow: 1px 1px 5px rgba(0,0,0,0.1);
}
.panel h4 {
    margin-top: 0px;
    margin-bottom: 6px;
}
@media (max-width: 700px) {
    .output-box {
        display: flex;
        flex-direction: column;
        align-items: center;
    }
    .panel {
        width: 95% !important;
    }
}
@media (min-width: 701px) {
    .output-box {
        display: flex;
        flex-direction: column;
        align-items: stretch;
    }
    .panel {
        width: 100% !important;
    }
}
</style>
"""))

# -------------------------------
# Helper functions (unchanged)
# -------------------------------
def get_uploaded_file_content(upload_widget):
    if not upload_widget.value:
        return None
    val = upload_widget.value
    if isinstance(val, dict):
        return list(val.values())[0]["content"]
    if isinstance(val, tuple):
        return val[0]["content"]
    return None

def normalize_image(img):
    img = img - np.min(img)
    if np.max(img) > 0:
        img = img / np.max(img)
    return (img * 255).astype(np.uint8)

def resize_if_needed(pil_img, max_size=512):
    if pil_img.width > max_size or pil_img.height > max_size:
        pil_img = pil_img.copy()
        pil_img.thumbnail((max_size, max_size))
    return pil_img

def to_grayscale_array(pil_img):
    if pil_img.mode != "L":
        pil_img = pil_img.convert("L")
    return np.array(pil_img, dtype=np.float32)

def circular_mask(shape, radius, filter_type):
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    Y, X = np.ogrid[:rows, :cols]
    dist = np.sqrt((X - ccol)**2 + (Y - crow)**2)
    if filter_type == 'Low-pass':
        mask = dist <= radius
    elif filter_type == 'High-pass':
        mask = dist >= radius
    else:
        mask = np.ones_like(dist, dtype=bool)
    return mask.astype(float)

def compute_radius_from_click(event, arr_shape):
    if event.xdata is None or event.ydata is None:
        return None
    rows, cols = arr_shape
    crow, ccol = rows // 2, cols // 2
    x, y = event.xdata, event.ydata
    r = np.sqrt((x - ccol)**2 + (y - crow)**2)
    return int(r)

# -------------------------------
# Widgets
# -------------------------------
title_html = widgets.HTML("<h2>Interactive Fourier Filtering Explorer</h2>")
subtitle_html = widgets.HTML("<p>Upload an image, select a filter, and explore the frequency-domain mask interactively.</p>")

upload = widgets.FileUpload(accept="image/*", multiple=False, description="📁 Upload Image")
filter_type = widgets.Dropdown(options=['None','Low-pass','High-pass'], value='Low-pass', description='Filter:')
radius_slider = widgets.IntSlider(value=40, min=1, max=400, step=1, description='Radius:')
cmap_selector = widgets.Dropdown(options=['magma','inferno','viridis','gray'], value='magma', description='FFT colormap:')
out_status = widgets.Output()
out_original = widgets.Output()
out_fft = widgets.Output()
out_filtered = widgets.Output()

# -------------------------------
# Processing function (unchanged logic)
# -------------------------------
def process_and_display(change=None):
    out_status.clear_output(wait=True)
    out_original.clear_output(wait=True)
    out_fft.clear_output(wait=True)
    out_filtered.clear_output(wait=True)

    content = get_uploaded_file_content(upload)
    if content is None:
        with out_status:
            print("Waiting for image upload...")
        return

    try:
        with out_status:
            print("Processing image...")

        pil_img = Image.open(io.BytesIO(content))
        pil_img = resize_if_needed(pil_img)
        gray_arr = to_grayscale_array(pil_img)

        # FFT
        F = np.fft.fft2(gray_arr)
        Fshift = np.fft.fftshift(F)
        magnitude = np.abs(Fshift)
        display_magnitude = np.log1p(magnitude)

        # Mask
        rad = radius_slider.value
        mask = circular_mask(gray_arr.shape, rad, filter_type.value)
        Fshift_filtered = Fshift * mask

        # Inverse FFT
        F_ishift = np.fft.ifftshift(Fshift_filtered)
        img_filtered = np.abs(np.fft.ifft2(F_ishift))
        img_filtered_norm = normalize_image(img_filtered)

        # ORIGINAL
        with out_original:
            plt.figure(figsize=(4,4))
            plt.imshow(pil_img if pil_img.mode in ("RGB","RGBA") else gray_arr, cmap='gray')
            plt.title("Original Image")
            plt.axis('off')
            plt.show()

        # FFT + Mask overlay
        with out_fft:
            fig, ax = plt.subplots(figsize=(4,4))
            ax.imshow(display_magnitude, cmap=cmap_selector.value)
            if filter_type.value != "None":
                overlay = np.zeros((*mask.shape,4))
                overlay[mask==1] = [1,1,0,0.35]
                ax.imshow(overlay)
            ax.set_title("FFT Magnitude (click to set radius)")
            ax.axis('off')

            def onclick(event):
                r = compute_radius_from_click(event, gray_arr.shape)
                if r is not None:
                    radius_slider.value = r
            fig.canvas.mpl_connect("button_press_event", onclick)
            plt.show()

        # FILTERED IMAGE
        with out_filtered:
            plt.figure(figsize=(4,4))
            plt.imshow(img_filtered_norm, cmap='gray')
            plt.title(f"Filtered Output ({filter_type.value})")
            plt.axis('off')
            plt.show()

    except Exception as e:
        with out_status:
            print("Error:", e)

# -------------------------------
# Widget observers
# -------------------------------
upload.observe(process_and_display, names='value')
filter_type.observe(process_and_display, names='value')
radius_slider.observe(process_and_display, names='value')
cmap_selector.observe(process_and_display, names='value')

# -------------------------------
# Layout - fully vertical & professional
# -------------------------------
control_panel = widgets.VBox([
    widgets.HTML("<h3>Controls</h3>"),
    widgets.Box([upload], layout=widgets.Layout(margin='0 0 8px 0')),
    widgets.Box([filter_type], layout=widgets.Layout(margin='0 0 8px 0')),
    widgets.Box([radius_slider], layout=widgets.Layout(margin='0 0 8px 0')),
    widgets.Box([cmap_selector], layout=widgets.Layout(margin='0 0 8px 0')),
    out_status
], layout=widgets.Layout(width='100%', align_items='stretch'))

output_panel = widgets.VBox([
    widgets.Box([out_original], layout=widgets.Layout(width='100%', margin='0 0 12px 0')),
    widgets.Box([out_fft], layout=widgets.Layout(width='100%', margin='0 0 12px 0')),
    widgets.Box([out_filtered], layout=widgets.Layout(width='100%', margin='0 0 12px 0')),
], layout=widgets.Layout(width='100%'))

display(widgets.VBox([
    title_html,
    subtitle_html,
    control_panel,
    widgets.HTML("<hr>"),
    output_panel
], layout=widgets.Layout(width='100%')))
